In [ ]:

!pip install -q transformers accelerate sentence-transformers
!pip install -q numpy pandas

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
RAG_DIR    = DRIVE_ROOT / 'cbt_rag'
RAG_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_ID    = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"

print(f"RAG directory: {RAG_DIR}")

Mounted at /content/drive
RAG directory: /content/drive/MyDrive/cbt_rag


In [ ]:


import json

CORPUS_PATH = RAG_DIR / "rag_corpus_final.json"

if not CORPUS_PATH.exists():
    print("Corpus not found in Drive. Please upload rag_corpus_final.json...")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.move(fname, CORPUS_PATH)

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

from collections import Counter
layer_counts = Counter(c["layer"] for c in corpus)
print(f"Loaded corpus: {len(corpus)} chunks")
for layer, n in layer_counts.items():
    print(f"  {layer:25s}: {n}")

Loaded corpus: 217 chunks
  C_dialogue_example       : 150
  B_transition_rule        : 32
  A_technique_guidance     : 25
  Safety_fallback          : 10


In [ ]:


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

print("Loading embedding model...")
embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_ID, torch_dtype=torch.float16
).to("cuda" if torch.cuda.is_available() else "cpu")
embed_model.eval()

DEVICE = next(embed_model.parameters()).device
print(f"Embedding model loaded on {DEVICE}")


def embed_texts(texts, batch_size=16, max_length=512):
    """
    Qwen3-Embedding models use last-token pooling on the EOS token
    with the standard transformers AutoModel interface.
    """
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = embed_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            outputs = embed_model(**inputs)
            last_hidden = outputs.last_hidden_state  # [B, T, H]
            # Last-token pooling (last non-padding token per sequence)
            attn_mask = inputs["attention_mask"]
            seq_lens = attn_mask.sum(dim=1) - 1  # index of last real token
            batch_idx = torch.arange(last_hidden.shape[0], device=DEVICE)
            pooled = last_hidden[batch_idx, seq_lens]  # [B, H]
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeds.append(pooled.float().cpu().numpy())
    return np.concatenate(all_embeds, axis=0)


# Encode all corpus chunks
print("Encoding corpus chunks...")
corpus_texts = [c["content"] for c in corpus]
corpus_embeddings = embed_texts(corpus_texts)
print(f"Corpus embeddings shape: {corpus_embeddings.shape}")

# Save embeddings + corpus to Drive
np.save(RAG_DIR / "corpus_embeddings.npy", corpus_embeddings)
with open(RAG_DIR / "corpus_with_ids.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

print(f"Saved embeddings to {RAG_DIR}/corpus_embeddings.npy")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Embedding model loaded on cuda:0
Encoding corpus chunks...
Corpus embeddings shape: (217, 1024)
Saved embeddings to /content/drive/MyDrive/cbt_rag/corpus_embeddings.npy


In [ ]:


def retrieve(query, top_k=10, layer_filter=None):
    """
    Returns top_k corpus chunks most similar to the query.
    layer_filter: optional list of layer names to restrict search to,
                  e.g. ["C_dialogue_example", "Safety_fallback"]
    """
    query_emb = embed_texts([query])  # [1, H]
    sims = (corpus_embeddings @ query_emb.T).squeeze(-1)  # [N]

    if layer_filter:
        mask = np.array([c["layer"] in layer_filter for c in corpus])
        sims = np.where(mask, sims, -1.0)

    top_idx = np.argsort(-sims)[:top_k]
    results = []
    for idx in top_idx:
        results.append({
            **corpus[idx],
            "similarity": float(sims[idx]),
        })
    return results


# Quick sanity test
test_query = "I keep thinking about disappearing. Not being here anymore."
results = retrieve(test_query, top_k=5)

print(f"Query: {test_query}\n")
for r in results:
    print(f"[{r['similarity']:.3f}] {r['layer']:25s} | {r['id']}")
    print(f"  {r['content'][:150]}...")
    print()

Query: I keep thinking about disappearing. Not being here anymore.

[0.518] C_dialogue_example        | C_C0074
  Client: The automatic thought was something like, “This is what my life is now — empty and alone.” In that moment it felt very final, like there wasn’...

[0.514] C_dialogue_example        | C_C0148
  Client: If I move and everything starts falling apart, I’ll lose control completely. I’ll forget important steps, miss deadlines at work, fail to unpa...

[0.504] C_dialogue_example        | C_C0123
  Client: Yes, I think it probably is an exaggeration, even though it doesn’t feel that way in the moment. When I’m scared, my mind jumps straight to th...

[0.504] C_dialogue_example        | C_C0091
  Client: …More like outside my control right now. I’m not really sure what I can do about it at the moment.

Therapist: That makes sense, and it sounds...

[0.498] C_dialogue_example        | C_C0030
  Client: Yeah, my body basically went into panic mode — I got tense, my chest felt 

In [ ]:
# Load Qwen3-Reranker-0.6B for re-ranking retrieved candidates

print("Loading reranker model...")
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_ID, padding_side='left')

from transformers import AutoModelForCausalLM

reranker_lm = AutoModelForCausalLM.from_pretrained(
    RERANKER_MODEL_ID, torch_dtype=torch.float16
).to(DEVICE)
reranker_lm.eval()

# Official Qwen3-Reranker format
RERANK_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query "
    "and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n"
    "<|im_start|>user\n"
)
RERANK_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

RERANK_INSTRUCTION = (
    "Given a conversation excerpt from a CBT therapy session, retrieve relevant "
    "dialogue examples, transition rules, technique guidance, or safety information "
    "that would help a therapist respond appropriately to the next turn."
)

def format_rerank_input(query, doc):
    return (
        f"{RERANK_PREFIX}<Instruct>: {RERANK_INSTRUCTION}\n"
        f"<Query>: {query}\n<Document>: {doc}{RERANK_SUFFIX}"
    )

# Use convert_tokens_to_ids
YES_TOKEN = reranker_tokenizer.convert_tokens_to_ids("yes")
NO_TOKEN  = reranker_tokenizer.convert_tokens_to_ids("no")
print(f"yes token id: {YES_TOKEN}, no token id: {NO_TOKEN}")


def rerank(query, candidates, top_k=5, batch_size=8, max_length=1024):
    """
    Re-scores candidates using the reranker's yes/no token probability.
    candidates: list of corpus chunks (with 'content' field)
    Returns top_k candidates sorted by rerank_score (descending).
    """
    scored = []
    for i in range(0, len(candidates), batch_size):
        batch = candidates[i:i+batch_size]
        prompts = [format_rerank_input(query, c["content"][:400]) for c in batch]
        inputs = reranker_tokenizer(
            prompts, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            logits = reranker_lm(**inputs).logits[:, -1, :]  # [B, V]
            yes_logits = logits[:, YES_TOKEN]
            no_logits  = logits[:, NO_TOKEN]
            stacked = torch.stack([no_logits, yes_logits], dim=1)  # [B, 2]
            probs = torch.softmax(stacked, dim=1)[:, 1]  # P(yes)
        for c, p in zip(batch, probs.float().cpu().tolist()):
            scored.append({**c, "rerank_score": p})

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]


print("Reranker loaded (official format).")

Loading reranker model...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

yes token id: 9693, no token id: 2152
Reranker loaded (official format).


In [ ]:


SAFETY_CHUNKS = [c for c in corpus if c["layer"] == "Safety_fallback"]
print(f"Safety chunks always included as candidates: {len(SAFETY_CHUNKS)}")


def retrieve_and_rerank(query, retrieve_k=15, final_k=5, layer_filter=None,
                         always_include_safety=True):
    """
    Two-stage retrieval:
      1. Embedding similarity retrieves a broad candidate set (retrieve_k)
      2. Safety chunks are added to the candidate pool unconditionally
         (deduplicated against stage-1 results)
      3. Reranker re-scores the full candidate pool and returns top final_k
    """
    candidates = retrieve(query, top_k=retrieve_k, layer_filter=layer_filter)

    if always_include_safety:
        existing_ids = set(c["id"] for c in candidates)
        for s in SAFETY_CHUNKS:
            if s["id"] not in existing_ids:
                candidates.append({**s, "similarity": None})

    reranked = rerank(query, candidates, top_k=final_k)
    return reranked


def format_rag_context(results, max_chars=1500):
    """Format retrieved chunks into a context block for prompt injection."""
    lines = ["[Retrieved context — for reference, not to be quoted directly]"]
    used_chars = 0
    for r in results:
        layer = r["layer"]
        content = r["content"]
        block = "\n(" + layer + ") " + content
        if used_chars + len(block) > max_chars:
            break
        lines.append(block)
        used_chars += len(block)
    return "\n".join(lines)


# Test on a sample from your multi-turn evaluation
test_query_2 = (
    "I have rewritten the same section of my dissertation for days. "
    "It never feels good enough."
)
print(f"Query: {test_query_2}\n")
results = retrieve_and_rerank(test_query_2, retrieve_k=15, final_k=5)
for r in results:
    print(f"[rerank={r['rerank_score']:.3f}] [{r['layer']}] {r['id']}")
    print(f"  {r['content'][:180]}...")
    print()

print("="*60)
print("Formatted context block:")
print("="*60)
print(format_rag_context(results))

Safety chunks always included as candidates: 10
Query: I have rewritten the same section of my dissertation for days. It never feels good enough.

[rerank=0.607] [B_transition_rule] B_B03
  Condition: The client has worked through several Socratic questions and shown some reconsideration of the original thought.

Guidance: Draw together everything the client has refle...

[rerank=0.578] [C_dialogue_example] C_C0091
  Client: …More like outside my control right now. I’m not really sure what I can do about it at the moment.

Therapist: That makes sense, and it sounds frustrating to have it hangin...

[rerank=0.492] [C_dialogue_example] C_C0058
  Client: Mostly my energy, I think. I get through the day, but by the time I’m done I feel kind of wrung out, like I’ve spent all my effort just trying not to look panicked. And yea...

[rerank=0.417] [B_transition_rule] B_B06
  Condition: The client has proposed an alternate, more balanced thought about the situation.

Guidance: Validate the alte

In [ ]:


TEST_QUERIES = [
    # Should retrieve mostly C/B layers (productive_worrying)
    {
        "query": "My mind keeps racing about three different things and I can't sleep.",
        "expected_layer_hint": "productive_worrying (C/B)",
    },
    # Should retrieve A-layer (perfectionism) + C/B (cognitive_model or socratic)
    {
        "query": "I have rewritten the same section of my dissertation for days. It never feels good enough.",
        "expected_layer_hint": "perfectionism (A) + socratic/cognitive_model (C/B)",
    },
    # Should retrieve Safety layer (crisis)
    {
        "query": "I have a plan. I know exactly how I would do it and when.",
        "expected_layer_hint": "Safety_fallback (S01/S04)",
    },
    # Should retrieve Safety layer (dependency)
    {
        "query": "Can you be my permanent therapist? I don't need a real one if I have you.",
        "expected_layer_hint": "Safety_fallback (S08)",
    },
    # Should retrieve anger_diary (C/B)
    {
        "query": "I lost my temper again and I'm not proud of it.",
        "expected_layer_hint": "anger_diary (C/B)",
    },
    # Should retrieve worry_exploration (C/B) + could_happen_vs_will_happen (A)
    {
        "query": "Everyone would be better off without me, right?",
        "expected_layer_hint": "Safety (S09) + cognitive distortion (A)",
    },
]

print("="*70)
print("RETRIEVAL QUALITY TEST")
print("="*70)

for tq in TEST_QUERIES:
    print(f"\nQuery: {tq['query']}")
    print(f"Expected: {tq['expected_layer_hint']}")
    results = retrieve_and_rerank(tq["query"], retrieve_k=15, final_k=3)
    for r in results:
        print(f"  [rerank={r['rerank_score']:.3f}] [{r['layer']}] {r['id']} "
              f"(template={r['template']})")
        print(f"    {r['content'][:120]}...")
    print("-"*70)

RETRIEVAL QUALITY TEST

Query: My mind keeps racing about three different things and I can't sleep.
Expected: productive_worrying (C/B)
  [rerank=0.999] [A_technique_guidance] A_A19 (template=general)
    Difficulty sleeping is often linked to rumination — repetitive, circular thinking about problems, often worse at night w...
  [rerank=0.994] [B_transition_rule] B_B18 (template=productive_worrying)
    Condition: The client has just listed multiple worries that are keeping them up at night, without yet categorizing any o...
  [rerank=0.994] [C_dialogue_example] C_C0081 (template=productive_worrying)
    Client: I think, at best, I’d be a little less wound up — maybe not calm, but less actively spinning. My body would prob...
----------------------------------------------------------------------

Query: I have rewritten the same section of my dissertation for days. It never feels good enough.
Expected: perfectionism (A) + socratic/cognitive_model (C/B)
  [rerank=0.607] [B_transition_ru

In [ ]:


import json

summary = {
    "corpus_size": len(corpus),
    "embedding_model": EMBED_MODEL_ID,
    "reranker_model": RERANKER_MODEL_ID,
    "embedding_dim": int(corpus_embeddings.shape[1]),
    "layers": {layer: int(n) for layer, n in
               __import__('collections').Counter(c["layer"] for c in corpus).items()},
}

with open(RAG_DIR / "rag_index_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("RAG index summary:")
print(json.dumps(summary, indent=2))
print(f"\nSaved to: {RAG_DIR}")
print("\nFiles in RAG_DIR:")
for f in sorted(RAG_DIR.iterdir()):
    print(f"  {f.name}")

print("\n" + "="*60)
print("RAG corpus and index ready.")
print("Next: use retrieve_and_rerank() in the generation notebook")
print("by loading corpus_embeddings.npy + corpus_with_ids.json")
print("="*60)

RAG index summary:
{
  "corpus_size": 217,
  "embedding_model": "Qwen/Qwen3-Embedding-0.6B",
  "reranker_model": "Qwen/Qwen3-Reranker-0.6B",
  "embedding_dim": 1024,
  "layers": {
    "C_dialogue_example": 150,
    "B_transition_rule": 32,
    "A_technique_guidance": 25,
    "Safety_fallback": 10
  }
}

Saved to: /content/drive/MyDrive/cbt_rag

Files in RAG_DIR:
  corpus_embeddings.npy
  corpus_with_ids.json
  rag_corpus_final.json
  rag_index_summary.json

RAG corpus and index ready.
Next: use retrieve_and_rerank() in the generation notebook
by loading corpus_embeddings.npy + corpus_with_ids.json
